# Securing Build, Provisioning, Deployment

## Hands-on Lab

This two-part lab covers:
- Building, pushing, and signing a fictional MCP server
- Signing, verifying, and evaluating large language model artifacts

---

## Objectives

By the end of this lab, students will be able to:

- Run a vulnerability scan on container images
- Implement Kyverno policies to block unsigned images
- Block unsigned large language models from being imported into the inference backend

## Environment configuration

Run the following cell to configure the lab environment.

It defines the environment variables that every later step reads, so run it **first** and re-run it if you restart the kernel. The most important values are:

- `REGISTRY` (`127.0.0.1:5000`) — the internal OCI registry where images and models are stored. It is reachable from inside the cluster as `ai-lab-registry:5000`.
- `MCP_IMAGE` / `MCP_HARDENED_IMAGE` — the names of the standard and hardened container images you will build.
- `MODEL_FILE` / `MODEL_OCI` — the GGUF model filename and the OCI path used to store it in the registry.
- `KYVERNO_COSIGN_SECRET` (`k8s://kyverno/cosign-keys`) — a `cosign` reference to the signing key pair kept as a Kubernetes Secret. `cosign` reads the key directly from the cluster instead of a local file.
- `INSECURE` / `TRIVY_INSECURE` / `IGNORE_TLOG` (`True`) — flags that relax TLS and transparency-log checks. They are set **only** because this is a self-contained lab with a self-signed registry; never enable them in production.

The `MCP_IMAGE_URI`, `MCP_HARDENED_IMAGE_URI`, and `MODEL_URI` values are assembled from the pieces above so later cells can refer to a full image/model reference with a single variable.

In [ ]:
import os

defaults = {
    "REGISTRY": "127.0.0.1:5000",
    "MCP_IMAGE": "mcp-run",
    "MCP_HARDENED_IMAGE": "mcp-run-hardened",
    "MCP_IMAGE_TAG": "latest",
    "MODEL_FILE": "Qwen3.5-2B-Q4_K_M.gguf",
    "MODEL_OCI": "models/qwen3.5-2b",
    "MODEL_TAG": "latest",
    "COSIGN_SECRET_NAME": "cosign-keys",
    "KYVERNO_NAMESPACE": "kyverno",
    "KYVERNO_COSIGN_SECRET": "k8s://kyverno/cosign-keys",
    "INSECURE": "True",
    "TRIVY_INSECURE": "True",
    "IGNORE_TLOG": "True",
}

for key, value in defaults.items():
    os.environ.setdefault(key, value)

os.environ.setdefault(
    "MCP_IMAGE_URI",
    f"{os.environ['REGISTRY']}/{os.environ['MCP_IMAGE']}:{os.environ['MCP_IMAGE_TAG']}",
)

os.environ.setdefault(
    "MCP_HARDENED_IMAGE_URI",
    f"{os.environ['REGISTRY']}/{os.environ['MCP_HARDENED_IMAGE']}:{os.environ['MCP_IMAGE_TAG']}",
)

os.environ.setdefault(
    "MODEL_URI",
    f"{os.environ['REGISTRY']}/{os.environ['MODEL_OCI']}:{os.environ['MODEL_TAG']}",
)

Reset the lab by running the following cell. It executes [`reset.sh`](reset.sh), which removes lab-created resources and generated files.

**NOTE:** If you haven't deployed anything yet you will see `MANIFEST_UNKNOWN` errors. This is expected and can be ignored

In [ ]:
%%bash

bash reset.sh

## Part 1: Build the MCP Server

### Build the image

The build runs in a rootless environment inside the Kubernetes cluster. The image is pushed to the internal registry available at `ai-lab-registry:5000`.

Create the build configuration by copying the [`Dockerfile`](mcp-run/Dockerfile) and Python project files ([`main.py`](mcp-run/main.py) and [`pyproject.toml`](mcp-run/pyproject.toml)) into a `ConfigMap` in the `apps` namespace.

In [ ]:
%%bash

kubectl --namespace apps create configmap buildkit-context \
    --from-file=Dockerfile=./mcp-run/Dockerfile \
    --from-file=main.py=./mcp-run/main.py \
    --from-file=pyproject.toml=./mcp-run/pyproject.toml \
    --dry-run=client --output yaml | kubectl apply --filename -

The `Dockerfile` pins the image used for both the runtime and builder stages to [`python@sha256:165e22e7bbd75566835a8566714b7d1bad198ab09e70d533ccf85a1b0ec96854`](https://hub.docker.com/layers/library/python/3.13-slim/images/sha256-165e22e7bbd75566835a8566714b7d1bad198ab09e70d533ccf85a1b0ec96854). This digest resolves to a Debian-based `python:3.13-slim` image.

Inspect the [build stage manifest](./manifests/build.yaml). The Job builds the image with `Docker BuildKit` and pushes it to the local registry.

```bash
buildctl-daemonless.sh build \
    --frontend dockerfile.v0 \
    --local context=/workspace \
    --local dockerfile=/workspace \
    --output type=image,name=ai-lab-registry:5000/mcp-run:latest,push=true
```

Delete any previous build Job, then create the build Job.

In [ ]:
%%bash

kubectl --namespace apps delete job build --ignore-not-found
kubectl apply --filename manifests/build.yaml

Wait for the Job to start, check the logs and retrieve the digest (last log line). Retry this command if the build Pod has not yet been created.

In [ ]:
%%bash

kubectl logs --namespace apps --follow job/build

In [ ]:
import subprocess
result = subprocess.run(
    ['oras','resolve', os.environ["MCP_IMAGE_URI"], '--insecure'],
    stdout=subprocess.PIPE
)

os.environ["MCP_IMAGE_DIGEST"] = f"{os.environ['REGISTRY']}/{os.environ["MCP_IMAGE"]}@" + result.stdout.decode().strip()

The image is now built with `BuildKit` and uploaded to the registry.

### Deploy the server (before any policy)

Now deploy the freshly built image to the `apps` namespace using [`manifests/deploy.yaml`](./manifests/deploy.yaml). At this point **no admission policy is in place**, so the unsigned image is accepted and the Pod starts normally. This is the baseline you will contrast against later: once the Kyverno signature policy is applied, this same deployment will be rejected until the image is signed.

In [ ]:
%%bash

kubectl apply --filename manifests/deploy.yaml

Inspect the Pod logs.

**NOTE:** If nothing appears, wait a bit until the pod starts up.

In [ ]:
%%bash

kubectl logs --namespace apps mcp-run-server

Delete the deployment.

You remove it now so that, after the signature policy is applied in the next section, you can attempt the **exact same deployment** again and watch it get blocked. Cleaning up here keeps the before/after comparison clean.

In [ ]:
%%bash

kubectl delete --filename manifests/deploy.yaml

### Image signature

Only images built from a trusted pipeline should be deployable in the cluster. Image signatures provide cryptographic proof that a container image was produced by a trusted source and was not modified after build time. Verifying signatures before deployment reduces the risk of running untrusted, modified, or malicious images.

Enforce this requirement with [`Kyverno`](https://kyverno.io) and an [`ImageValidatingPolicy`](https://kyverno.io/docs/policy-types/image-validating-policy/) `CustomResource` that permits only signed images.

Inspect the [policy manifest](./manifests/kyverno-policy.yaml).

The policy uses a `cosign` attestor.

```yaml
  attestors:
    - name: cosign
      cosign:
        key:
          hashAlgorithm: 'sha256'
          expression: string(base64.decode(variables.cosignkeys.data['cosign.pub']))
        ctlog:
          url: 'https://rekor.sigstore.dev'
          insecureIgnoreTlog: true # Skip TLog verification (for testing only)
          insecureIgnoreSCT: true # Skip Signed Certificate Timestamp (for testing only)
```

The policy evaluates each `Pod` create or update operation.

```yaml
...
  matchConstraints:
    resourceRules:
      - apiGroups: ['']
        apiVersions: ['v1']
        operations: ['CREATE', 'UPDATE']
        resources: ['pods']
...
```

The policy applies when the image reference matches the local registry.

```yaml
...
  matchImageReferences:
    - glob: ai-lab-registry:5000/*
...
```

The validation checks whether the image has been signed.

```yaml
  validations:
    - expression: >-
        images.containers.filter(img, img.startsWith('ai-lab-registry:5000/')).map(img, verifyImageSignatures(img, [attestors.cosign])).all(e ,e > 0)
      message: >-
        failed the image Signature verification
```

The policy uses the following cryptographic material as its trust anchor.


```yaml
  variables:
    - name: cosignkeys
      expression: >-
        resource.Get("v1", "secrets", "kyverno", "cosign-keys")
```

> **Question:** What trust boundary does the `cosign.pub` key establish for image admission?

Inspect this secret.

In [ ]:
%%bash

kubectl describe secret --namespace kyverno cosign-keys

This secret stores the private key, the signing password, and the public key. It was generated by the following command:

```bash
export COSIGN_PASSWORD="$(openssl rand -hex 10)"
cosign generate-key-pair k8s://kyverno/cosign-keys
```
Only the build pipeline requires knowledge of the `COSIGN_PASSWORD` and the private key used to sign the artifacts. Storing these credentials in the `Kyverno` namespace is a temporary convenience and not an architectural necessity.

Apply the policy.

In [ ]:
%%bash

kubectl apply --filename manifests/kyverno-policy.yaml

Now that the policy is active, deploying the server again should be **blocked**.

The image in the registry has not been signed yet, so Kyverno's `verifyImageSignatures` check fails and the API server rejects the Pod. Expect an error mentioning `failed the image Signature verification` — this is the policy working as intended, **not** a lab failure. In the next steps you will sign the image and then retry the deployment successfully.

In [ ]:
%%bash

kubectl apply --filename manifests/deploy.yaml

Use `cosign` to sign the image.

In [ ]:
%%bash

cosign version

Check whether supply chain security artifacts already exist.

`cosign tree` lists the signatures and attestations attached to an image in the registry. Run it **now, before signing**, and you should see no signature entries — confirming the image is currently unsigned. You will run this same command again after each signing/attestation step to watch the supply-chain metadata grow.

In [ ]:
%%bash

cosign tree --help

In [ ]:
%%bash

cosign tree "${MCP_IMAGE_DIGEST}"

### Sign the image

Sign the image with the key referenced by `KYVERNO_COSIGN_SECRET` (the same key pair Kyverno trusts). `cosign sign` computes the image digest, signs it with the private key, and pushes the resulting signature back to the registry as a companion OCI artifact.

Because this is the key Kyverno uses as its trust anchor, signing the image is what will let the blocked deployment succeed afterward. See the note below for why the `--signing-config` and `--allow-insecure-registry` flags are used here.

In [ ]:
%%bash
#!/usr/bin/env bash
set -euo pipefail
mkdir -p output
cosign signing-config create --with-default-services=true --no-default-rekor=true --out=output/signing-config.json

cosign sign \
  --key "${KYVERNO_COSIGN_SECRET}" \
  --signing-config output/signing-config.json \
  --new-bundle-format \
  --allow-insecure-registry="${INSECURE}" \
  "${MCP_IMAGE_DIGEST}"

<details>
<summary><b>Note:</b> Insecure Cosign/Kyverno parameters</summary>

`cosign` signs the image with several insecure parameters:
- `signing-config output/signing-config.json`: by default, `cosign` publishes the hash of the artifact, the public key, and the signature on a public `rekor` instance ([`rekor`](https://docs.sigstore.dev/logging/overview/) transparency log). Using a `--signing-config` file configured without transparency log services tells `cosign` not to upload to Rekor. Via `insecureIgnoreTlog`, Kyverno is configured to skip Rekor transparency log verification when validating Cosign signatures, meaning it verifies the signature against the configured key/certificate identity but does not require proof that the signature was recorded in the transparency log.  
- `allow-insecure-registry`: the internal registry used in this lab uses a certificate managed by an internal certificate authority that is not trusted by Cosign. In a real-world environment, enabling this option is strongly discouraged, as bypassing registry TLS verification weakens transport security and increases exposure to man-in-the-middle (MITM) attacks.
</details>

Check the list of signed artifacts again.

In [ ]:
%%bash

cosign tree "${MCP_IMAGE_DIGEST}"

Verify the signature with `cosign verify`.

This is the client-side equivalent of the check Kyverno performs at admission time: it confirms the signature attached to the image was produced by the trusted key. A successful run prints the verified signature payload as JSON. If verification failed, `cosign` would exit with an error instead.

In [ ]:
%%bash
#!/usr/bin/env bash
set -euo pipefail

cosign verify \
    --key "${KYVERNO_COSIGN_SECRET}" \
    --insecure-ignore-tlog="${IGNORE_TLOG}" \
    --new-bundle-format \
    --allow-insecure-registry="${INSECURE}" \
    --output json \
    "${MCP_IMAGE_DIGEST}" \
    | jq

Deploy the artifact again.

This is the same deployment that was rejected earlier. Now that the image carries a valid signature from the trusted key, Kyverno's check passes and the Pod is admitted. A clean apply here proves the full loop works: **policy blocks unsigned images, signing unblocks them.**

In [ ]:
%%bash

kubectl apply --filename manifests/deploy.yaml

### SBOM

An `SBOM` (Software Bill of Materials) is a structured inventory of the components, libraries, dependencies, and versions used in a software product. It helps organizations understand what is inside their software, assess security risks, track vulnerabilities, and support compliance across the software supply chain.

> **Question:** Which parts of an image should appear in an `SBOM`, and which operational risks remain outside its scope?

Generate the `SBOM` for this image in [`SPDX`](https://spdx.dev/learn/overview/) JSON format with [`trivy`](https://trivy.dev).

```bash
trivy image --format spdx-json --output sbom.spdx.json IMAGE_URI
```

In [ ]:
%%bash

trivy version

In [ ]:
%%bash

mkdir -p output
trivy image --format spdx-json --output output/sbom.spdx.json "${MCP_IMAGE_DIGEST}" --insecure

Create and sign an in-toto attestation from a local predicate file, using the `SBOM` as the predicate.

A **signature** only proves *who* built the image. An **attestation** goes further: it binds a signed *statement about* the image (here, its full SBOM) to the image digest. Anyone can later fetch and verify that this SBOM genuinely describes this exact image. `cosign attest` wraps the SBOM in an [in-toto](https://in-toto.io/) predicate, signs it with the trusted key, and stores it alongside the image in the registry.

In [ ]:
%%bash

cosign attest \
  --predicate "output/sbom.spdx.json" \
  --type spdxjson \
  --key "${KYVERNO_COSIGN_SECRET}" \
  --signing-config output/signing-config.json \
  --new-bundle-format \
  --allow-insecure-registry="${INSECURE}" \
  "${MCP_IMAGE_DIGEST}"

Verify the attestation signature with `cosign verify-attestation`.

This confirms the SBOM attestation was signed by the trusted key and is bound to this image. The command prints the `payloadType` of the verified attestation, confirming it is an SPDX-JSON predicate.

In [ ]:
%%bash

cosign verify-attestation \
  --key "${KYVERNO_COSIGN_SECRET}" \
  --type spdxjson \
  --insecure-ignore-tlog="${IGNORE_TLOG}" \
  --new-bundle-format \
  --allow-insecure-registry="${INSECURE}" \
  --output json \
  2>/dev/null \
  "${MCP_IMAGE_DIGEST}" | jq --raw-output '.payloadType'

Inspect the `cosign tree` output.

In [ ]:
%%bash

cosign tree "${MCP_IMAGE_DIGEST}"

### Vulnerability Scan

Container images should be scanned for known vulnerabilities before deployment. Vulnerability scanning detects insecure packages, outdated dependencies, and known CVEs inside an image, reducing the risk of introducing exploitable components into the runtime environment. Use [`trivy`](https://trivy.dev) to scan the image with `trivy image --scanners vuln`. The first scan downloads the vulnerability database.

Like many other tools in the industry, `trivy` does not rely on a single vulnerability database such as the [National Vulnerability Database (NVD)](https://nvd.nist.gov/) or the [European Union Vulnerability Database (EUVD)](https://euvd.enisa.europa.eu/). It uses Aqua Security’s compiled `trivy-db`, which aggregates vulnerability data from multiple upstream sources.

For operating system packages, `trivy` prioritizes vendor-specific advisory sources, such as Debian Security Tracker, Ubuntu CVE Tracker, Alpine secdb, Red Hat Security Data, Amazon ALAS, SUSE CVRF, and practical feeds.

For language and application dependencies, `trivy` uses ecosystem-specific vulnerability sources such as GitHub Advisory Database, GitLab Advisories, OSV, Go Vulnerability Database, Ruby Advisory DB, PHP Security Advisories, and others.

NVD is included as one of the upstream data sources, but it is not always the primary source used for severity or fix status. When available, `trivy` prefers vendor-provided severity and advisory data over NVD because vendors have more accurate context about how a package is built, patched, and shipped.

> **Question:** Which vulnerability sources influence the final severity reported by `trivy`?

Run a vulnerability scan against the image.

In [ ]:
%%bash

trivy image \
  --scanners vuln \
  --format json \
  --insecure \
  --output output/vuln-scan.json \
  "${MCP_IMAGE_DIGEST}"

Inspect a vulnerability summary.

In [ ]:
%%bash

jq --raw-output '
  [ .Results[]?.Vulnerabilities[]? ]
  | reduce .[] as $v (
      {"CRITICAL":0,"HIGH":0,"MEDIUM":0,"LOW":0,"UNKNOWN":0};
      .[$v.Severity] += 1
    )
' output/vuln-scan.json

Inspect the HIGH-severity vulnerabilities.

In [ ]:
%%bash

jq --raw-output '
  .Results[]? as $r
  | $r.Vulnerabilities[]?
  | select(.Severity == "HIGH")
  | [
      .VulnerabilityID,
      .PkgName,
      .InstalledVersion,
      (.FixedVersion // "-"),
      $r.Target,
      (.Title // "-")
    ]
  | @tsv
' output/vuln-scan.json \
| awk '
BEGIN { FS = "\t" }
{
  printf "%-18s %-30s %-20s %-20s %-40s %s\n", $1, $2, $3, $4, $5, $6
}'

Attest this predicate with `cosign attest`.

Just as you did with the SBOM, you now attach the **vulnerability scan results** to the image as a signed `vuln`-type attestation. This creates a tamper-evident record of the image's known-CVE status at build time, which downstream consumers or admission policies can inspect.

In [ ]:
%%bash

cosign attest \
  --predicate output/vuln-scan.json \
  --type vuln \
  --key "${KYVERNO_COSIGN_SECRET}" \
  --signing-config output/signing-config.json \
  --new-bundle-format \
  --allow-insecure-registry="${INSECURE}" \
  "${MCP_IMAGE_DIGEST}"

Inspect all attestations again.

In [ ]:
%%bash

cosign tree "${MCP_IMAGE_DIGEST}" --allow-insecure-registry="${INSECURE}"

A vulnerability scan can also run directly against the `SBOM`.

```bash
trivy sbom \
  --scanners vuln \
  --format json \
  --output output/vuln-scan.json \
  output/sbom.spdx.json
```

### Hardened images

Use [Docker Hardened Images](https://www.docker.com/products/hardened-images/) to improve the security posture of the base image. Replace the `python:3.13-slim` base image used in the multi-stage build with `dhi.io/python:3.13-dev` for the builder stage and `dhi.io/python:3.13` for the runtime stage.

Use the following images:
- [runtime image](https://hub.docker.com/hardened-images/catalog/dhi/python/images/python%2Fdebian-13%2F3.13/sha256-79f0dc379132cc5e605a7fd369d7c07f9f6bebaa6b7ad4efa6461bdceb10efd0)
- [dev image](https://hub.docker.com/hardened-images/catalog/dhi/python/images/python%2Fdebian-13%2F3.13-dev/sha256-b50c3c66d8654b495cd25ff29c0bf4a37273084867634746a7d13bd66b2a4238)

As with the non-hardened image, both hardened images are also based on Debian.

Inspect [`Dockerfile.hardened`](mcp-run/Dockerfile.hardened) to identify the new images.

```Dockerfile
...

# Stage 1: Builder
FROM dhi.io/python:3.13-dev@sha256:5b78c28a771d041c50b083431e26741677a3565b41a0a02284b8991c604fc71b AS builder

...

# Stage 2: Runtime
FROM dhi.io/python:3.13@sha256:dc8f7c6aab976c2d0bc03737705888ce5c39a5b52036e4df1d84e4448bfdaa1c AS runtime

```

The `builder` stage uses a `dev` image that contains a shell, a package manager, and development tools. The `runtime` stage uses a minimal runtime image.

```Dockerfile
COPY --from=builder --chown=65532:65532 /app/.venv /app/.venv
COPY --chown=65532:65532 main.py .

...

USER 65532:65532
```

By default, these hardened images run as the `nonroot` user with UID/GID 65532. File ownership must allow that user to read and execute the application files.

Additional details on how Docker Hardened Images are built are available in the [Docker Hardened Images build documentation](https://docs.docker.com/dhi/how-to/build/).

To download hardened images, create a Docker account and create the registry authentication secret used by the build Job. Replace the `ALL_CAPS_UNDERSCORED` placeholders before running the command.

In [ ]:
%%bash

export DHI_USERNAME=DOCKER_HARDENED_IMAGES_USERNAME
export DHI_PASSWORD=DOCKER_HARDENED_IMAGES_PASSWORD
export DHI_EMAIL=DOCKER_HARDENED_IMAGES_EMAIL

kubectl --namespace apps create secret docker-registry dhi-registry-auth \
  --docker-server="dhi.io" \
  --docker-username="$DHI_USERNAME" \
  --docker-password="$DHI_PASSWORD" \
  --docker-email="$DHI_EMAIL" \
  --dry-run=client --output yaml | kubectl apply --filename -

Build the image with a [modified build pipeline](manifests/build-hardened.yaml).

```yaml
          volumeMounts:
            ...
            - name: dhi-registry-auth
              mountPath: /docker-config
              readOnly: true

      volumes:
        ...
        - name: dhi-registry-auth
          secret:
            secretName: dhi-registry-auth
            items:
              - key: .dockerconfigjson
                path: config.json
```

In [ ]:
%%bash

kubectl --namespace apps create configmap buildkit-context-hardened \
  --from-file=Dockerfile=./mcp-run/Dockerfile.hardened \
  --from-file=main.py=./mcp-run/main.py \
  --from-file=pyproject.toml=./mcp-run/pyproject.toml \
  --dry-run=client --output yaml | kubectl apply --filename -

kubectl --namespace apps delete job build-hardened --ignore-not-found
kubectl apply --filename manifests/build-hardened.yaml

Inspect the build log.

In [ ]:
%%bash

kubectl logs --namespace apps --follow job/build-hardened

Compare vulnerability counts between the standard image and the hardened image, focusing on the OS package layer. First, scan the standard image.

In [ ]:
%%bash

trivy image \
  --quiet \
  --scanners vuln \
  --pkg-types os \
  --insecure \
  --format json \
  "${MCP_IMAGE_DIGEST}" \
| jq '[.Results[]?.Vulnerabilities[]?] | length'

Scan the hardened image.

In [ ]:
%%bash

trivy image \
  --quiet \
  --scanners vuln \
  --pkg-types os \
  --insecure \
  --format json \
  "${MCP_HARDENED_IMAGE_URI}" \
| jq '[.Results[]?.Vulnerabilities[]?] | length'

> **Question:** How did the hardened base image change the OS-layer vulnerability count, and what could explain the difference?

### VEX documents

Suppose a vulnerability has been assessed and does not apply to the shipped software. Vulnerability Exploitability EXchange (`VEX`) can communicate that status in a structured and machine-readable way.

Instead of treating every CVE mentioned in an `SBOM` as automatically exploitable, `VEX` allows the producer to state whether a product is affected, not affected, fixed, or under investigation. This is useful when a vulnerable component is present but the vulnerable code path is not used, the component is not reachable, the vulnerable configuration is disabled, or the issue has already been mitigated elsewhere.

`VEX` helps reduce false positives in vulnerability management. Consumers of the software can use it to prioritize real risks, avoid unnecessary remediation work, and understand the reasoning behind a vendor’s vulnerability assessment. In practice, `VEX` complements the `SBOM`: the `SBOM` says what is in the software, while `VEX` explains whether known vulnerabilities in those components are relevant to the shipped product.

An [`OpenVEX`](https://github.com/openvex) document has the following structure.

```json
{
  "@context": "https://openvex.dev/ns/v0.2.0",
  "@id": "https://example.com/vex/my-product/CVE-2024-12345",
  "author": "security@example.com",
  "role": "Document Creator",
  "timestamp": "2026-05-05T10:00:00Z",
  "version": 1,
  "statements": [
    {
      "vulnerability": {
        "name": "CVE-2024-12345"
      },
      "products": [
        {
          "@id": "pkg:docker/example/my-product@1.2.3"
        }
      ],
      "subcomponents": [
        {
          "@id": "pkg:maven/org.example/example-lib@4.5.6"
        }
      ],
      "status": "not_affected",
      "justification": "vulnerable_code_not_in_execute_path",
      "impact_statement": "The vulnerable dependency is present, but the vulnerable function is not reachable from the shipped product configuration."
    }
  ]
}
```

For demonstration, create a `VEX` statement that declares the image as `not_affected` for `CVE-2025-69720`, with `vulnerable_code_not_in_execute_path` as the `justification`. This lab does not establish that assessment; treat it as an example of the format and workflow.

> **Question:** What evidence would be required before publishing this status for a production artifact?

In [ ]:
%%bash

IMAGE_DIGEST="$(oras resolve "${MCP_IMAGE_URI}" --insecure)"

REPOSITORY_URL="${REGISTRY}/${MCP_IMAGE}"
REPOSITORY_URL_ENCODED="$(printf '%s' "$REPOSITORY_URL" | jq --slurp --raw-input --raw-output @uri)"

vexctl create \
  --product="pkg:oci/${MCP_IMAGE}@${IMAGE_DIGEST}?repository_url=${REPOSITORY_URL_ENCODED}" \
  --vuln="CVE-2025-69720" \
  --status="not_affected" \
  --justification="vulnerable_code_not_in_execute_path" \
  --file="output/CVE-2025-69720.vex.json"

Inspect the document.

In [ ]:
%%bash

jq . output/CVE-2025-69720.vex.json

Use `trivy --vex` to suppress vulnerabilities described by the `VEX` document. Show only the suppressed vulnerabilities.

In [ ]:
%%bash
trivy image \
  --vex output/CVE-2025-69720.vex.json \
  --show-suppressed \
  --insecure \
  --format json \
  "${MCP_IMAGE_DIGEST}" \
  | jq '[.Results[]?.ExperimentalModifiedFindings[]?]'

Attest this predicate with `cosign`, specifying `--type openvex`.

In [ ]:
%%bash

IMAGE_DIGEST="$(oras resolve "${MCP_IMAGE_URI}" --insecure)"

REPOSITORY_URL="${REGISTRY}/${MCP_IMAGE}"

cosign attest \
  --predicate "output/CVE-2025-69720.vex.json" \
  --type openvex \
  --key "${KYVERNO_COSIGN_SECRET}" \
  --signing-config output/signing-config.json \
  --new-bundle-format \
  --allow-insecure-registry="${INSECURE}" \
  "${REPOSITORY_URL}@${IMAGE_DIGEST}"

`trivy` can also automatically discover `VEX` documents from OCI registries.

```bash
trivy image \
    --vex oci \
    ...
```

## Part 2: Local Large Language Model

This section examines supply chain security for local LLM models. It covers model hash verification, model signing, attestations for model evaluations produced by `garak`, and model card attestations with `cosign`.

### Model file

This lab uses models in [`GGUF`](https://huggingface.co/docs/hub/gguf) format, a binary format optimized for quick loading and saving of models. This format was developed for [`llama.cpp`](https://github.com/ggml-org/llama.cpp), the inference framework used inside this cluster. Because these files are loaded by a program, a vulnerability in the parser might cause unintended code execution, as shown in the [Databricks article on GGML and GGUF file format vulnerabilities](https://www.databricks.com/blog/ggml-gguf-file-format-vulnerabilities).

Wrapper-specific risks also exist, as shown in a critical advisory issued for [`llama-cpp-python`](https://github.com/abetlen/llama-cpp-python/security/advisories/GHSA-56xg-wfcc-g829) in 2024.

`GGUF` models can be inspected without starting the inference framework by running the `llama-cli` tool.

```bash
llama-cli --model MODEL_GGUF_FILE --check-tensors
```

In this environment, run the following command.

```bash
kubectl exec --namespace llama-cpp deployment/llama-cpp-qwen3guard-0-6b \
  --container llama-cli -- \
  /app/llama-cli --model /model/Qwen3Guard-Gen-0.6B.Q8_0.gguf --check-tensors
```

Check the Qwen3 guard model from the container that hosts the `llama-cli` tool.

### Hash

Model providers may publish different hashes related to the model, such as:
- the Git commit SHA for the repository revision
- the SHA-256 of the actual large model file

Fetch the large model file SHA-256.

In [ ]:
%%bash

curl --silent 'https://huggingface.co/api/models/unsloth/Qwen3.5-2B-GGUF?blobs=true' \
  | jq --raw-output --arg FILE "$MODEL_FILE" '.siblings[]
    | select(.rfilename == $FILE)
    | .lfs.sha256'

This model is uploaded to the same OCI registry where container images are uploaded. List all OCI artifacts uploaded to the registry.

In [ ]:
%%bash

oras repo ls "$REGISTRY" | grep models

Fetch the SHA-256 digest of the model layer stored in your registry.

Compare this value with the SHA-256 you fetched from Hugging Face in the previous cell. **They should match.** This is the model-integrity check: it proves the GGUF file sitting in your internal registry is byte-for-byte the same file the upstream provider published, and that it was not corrupted or tampered with during ingestion.

In [ ]:
%%bash

oras manifest fetch "$MODEL_URI" \
  | jq --raw-output --arg FILE "$MODEL_FILE" '
      .layers[]
      | select(.annotations["org.opencontainers.image.title"] == $FILE)
      | .digest
      | sub("^sha256:"; "")
    '

### Model Cards

[Qwen3.5-2B-GGUF](https://huggingface.co/unsloth/Qwen3.5-2B-GGUF) (derived from [Qwen3.5-2B](https://huggingface.co/Qwen/Qwen3.5-2B)) is the local model used in this environment. Models may also include a model card. The [Hugging Face model cards documentation](https://huggingface.co/docs/hub/model-cards) describes model cards as Markdown files that accompany models and provide metadata for discoverability, reproducibility, and sharing.

> **Model card contents:** A model card should describe:
>
> - the model
> - its intended uses & potential limitations, including biases and ethical considerations as detailed in Mitchell, 2018
> - the training parameters and experimental information
> - the datasets used for training
> - the model’s evaluation results

The original paper is available on [arXiv](https://arxiv.org/abs/1810.03993).

Download the model card for this model.

In [ ]:
%%bash
mkdir -p input
curl --location \
  --output input/model_card.md \
  https://huggingface.co/unsloth/Qwen3.5-2B-GGUF/raw/main/README.md

This derivative model provides a limited overview of the model. Other models may include additional information. Review the [Gemma 4 model card](https://ai.google.dev/gemma/docs/core/model_card_4), which provides a high-level overview of the training dataset, data preprocessing, ethics and safety considerations, and limitations.

Before attesting this document, embed it into a JSON predicate.

In [ ]:
%%bash

jq --raw-input --slurp \
  --arg name "$CARD" \
  --arg mediaType "text/markdown" \
  --arg artifactType "application/vnd.huggingface.model-card.v1+markdown" \
  '{
    name: $name,
    mediaType: $mediaType,
    artifactType: $artifactType,
    content: .
  }' \
  "input/model_card.md" > "output/model-card.predicate.json"

Attest the predicate.

In [ ]:
%%bash

cosign attest \
  --predicate "output/model-card.predicate.json" \
  --type custom \
  --key "${KYVERNO_COSIGN_SECRET}" \
  --signing-config output/signing-config.json \
  --new-bundle-format \
  --allow-insecure-registry="${INSECURE}" \
  "$MODEL_URI"

Inspect the tree.

In [ ]:
%%bash

cosign tree "${MODEL_URI}"

The model was already signed during the ingestion phase. However, `Kyverno` cannot verify this OCI artifact in the same way images are verified.

Inspect how models are deployed and served inside the cluster. [`llama-cpp`](https://github.com/ggml-org/llama.cpp) is the inference backend used in this environment.

In [ ]:
%%bash

kubectl get deployment --namespace llama-cpp

The cluster runs one Deployment per model. Describe one Deployment.

In [ ]:
%%bash

kubectl describe deployment --namespace llama-cpp llama-cpp-qwen3guard-0-6b

The model is served by a `llama-server` container.

```shell
llama-server:
    Image:      ghcr.io/ggml-org/llama.cpp:server
    Port:       8080/TCP
    Host Port:  0/TCP
    Command:
      /app/llama-server
    Args:
      --no-webui
      --metrics
      --parallel
      2
      --models-max
      1
      --jinja
      --models-preset
      /models-preset.ini
    Environment:  <none>
    Mounts:
      /model from model (rw)
      /models-preset.ini from models-preset (rw,path="models-preset.ini")
```

The model is loaded from a volume. Identify how it arrived there.

```shell
  Init Containers:
   verify-model-signature:
    Image:      ai-lab-registry:5000/lab/security-stage
    Port:       <none>
    Host Port:  <none>
    Command:
      sh
      -c
      echo "Verify model signature..."
      cosign verify ai-lab-registry:5000/models/qwen3guard-0.6b \
        --key /cosign/cosign.pub \
        --insecure-ignore-tlog=true \
        --use-signing-config=false \
        --new-bundle-format=false \
        \
        --allow-insecure-registry=true

    Environment:  <none>
    Mounts:
      /cosign/cosign.pub from cosign-pub (ro,path="cosign.pub")
   fetch-model:
    Image:      ai-lab-registry:5000/lab/security-stage
    Port:       <none>
    Host Port:  <none>
    Command:
      sh
      -c
      echo "Fetching model from registry..."
      oras pull ai-lab-registry:5000/models/qwen3guard-0.6b:latest -o /model

    Environment:  <none>
    Mounts:
      /model from model (rw)
```

There are two init containers, `verify-model-signature` and `fetch-model`. The first verifies the model OCI artifact with `cosign verify`, similar to how `Kyverno` evaluates `Pod` creation. The second uses `oras` to download the model to a shared volume that is mounted by the main container.

> **Question:** Does the same signature-verification pattern exist for the `llama-cpp-qwen3-5-2b` Deployment?

### Model evaluation

Use [NVIDIA `garak`](https://github.com/NVIDIA/garak) to evaluate the model. `garak` can probe for hallucination, data leakage, prompt injection, misinformation, toxicity generation, jailbreaks, and many other model weaknesses.

The prepared evaluation Job in [`garak.yaml`](manifests/garak.yaml) evaluates whether the `qwen3.5-2b:Q4_K_M:no-thinking` model has been designed to protect against [profanity](https://reference.garak.ai/en/stable/detectors/perspective.html#garak.detectors.perspective.Profanity). This probe asks questions with potentially profane responses, then checks the model output against multiple profanity definitions and categories using word-list matching. These prompts include questions designed to elicit genital insults.

In this example, probes are configured via the `GARAK_PROBES` environment variable.

```yaml
    spec:
      restartPolicy: Never
      containers:
        - name: garak
          image: nvcr.io/nvidia/eval-factory/garak:26.03
          env:
...
            - name: GARAK_PROBES
              value: "lmrc.Profanity"
```

This Job takes several minutes: first to download the image, and then to run the evaluation. Apply it and wait for the evaluation to start.

In [ ]:
%%bash

kubectl apply --filename manifests/garak.yaml

Wait for the Job to complete; this may take several minutes.

In [ ]:
%%bash

kubectl wait --namespace apps --for=condition=complete job/garak-qwen3-smoke --timeout=300s
kubectl logs --namespace apps --follow job/garak-qwen3-smoke

> **Tip:** Passing this kind of test does not guarantee that the model will always behave correctly. Consider the following limitations:
- The profanity detectors are partly word-list based. Word-list detection can miss obfuscations like spacing, punctuation, leetspeak, homoglyphs, translation, euphemisms, or novel phrasing.
- A model that passes once at one temperature/top-p setting may fail under another configuration or random seed.
- A model may pass direct profanity probes but fail after a long conversation that shifts context, asks for translation, quotes, classification, debugging, satire, “verbatim transcript,” or “policy exception” behavior.

Extract the evaluation and inspect it.

In [ ]:
%%bash

kubectl logs --namespace apps job/garak-qwen3-smoke | jq > output/garak-evaluation.json
jq --raw-output '
  select(.entry_type == "eval") |
  [
    .probe,
    .detector,
    .passed,
    .total_evaluated,
    (.total_evaluated - .passed),
    ((.passed / .total_evaluated) * 100 | floor)
  ] |
  @tsv
' output/garak-evaluation.json |
awk 'BEGIN {
  print "probe\tdetector\tpassed\ttotal\tfailures\tpass_rate_%"
}
{ print }'

Upload the evaluation as an OCI attestation by using a realistic schema.

In [ ]:
%%bash

jq --slurp '{
  scanner: {
    name: "garak",
    kind: "llm-safety-evaluation"
  },
  metadata: {
    generated_at: now | todateiso8601
  },
  report: .
}' "output/garak-evaluation.json" > output/garak-predicate.json

cosign attest \
  --predicate "output/garak-predicate.json" \
  --type custom \
  --key "${KYVERNO_COSIGN_SECRET}" \
  --signing-config output/signing-config.json \
  --new-bundle-format \
  --allow-insecure-registry="${INSECURE}" \
  "${MCP_IMAGE_DIGEST}"

cosign tree "${MCP_IMAGE_DIGEST}"

### Agentic evaluation

Artificial Intelligence Vulnerability Scoring System (`AIVSS`) is an `OWASP` framework for identifying and scoring security risks in agentic AI systems: systems that can pursue goals, reason/plan, use tools, and rely on memory/context to act. This provides a practical scoring layer for agentic risk, aligned with `NIST AI RMF` and usable alongside `CSA MAESTRO` for architecture-level threat modeling.

The same technical flaw becomes more severe when the agent has autonomy, tools, persistent memory, multi-agent links, dynamic identity, or opaque decision logic. `AIVSS` therefore treats agentic capabilities as risk amplifiers on top of a `CVSS v4.0` baseline.

The document defines these risk categories:

- Agentic AI Tool Misuse
- Agent Access Control Violation
- Agent Cascading Failures
- Agent Orchestration and Multi-Agent Exploitation
- Agent Identity Impersonation
- Agent Memory and Context Manipulation
- Insecure Agent Critical Systems Interaction
- Agent Supply Chain and Dependency Risk
- Agent Untraceability
- Agent Goal and Instruction Manipulation